### `test_etd_solvers.ipynb` 
*Created: Sept 22, 2026* <br/>
Notebook for testing some Exponential Time Differencing (ETD) solvers by comparing them against trusted reference solvers from `OrdinaryDiffEq.jl`. We also test against exact solutions, when available.

In [1]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
using OrdinaryDiffEqExponentialRK, SciMLOperators 
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [10]:
#Import ETD test problems
@nbinclude("etd_test_problems.ipynb")

#Import ETD solvers
@nbinclude("etd_euler.ipynb")
@nbinclude("etd_rk2.ipynb")
@nbinclude("etd_rk3.ipynb")
@nbinclude("etd_rk4.ipynb")

#Importing testing tools 
@nbinclude("../../../tests.ipynb")

#Use desired plotting defaults for Makie 
@nbinclude("../../../../../../set_makie_defaults.ipynb")

In [3]:
#Convenience functions to allow passing the ETD solver a problem instance
etd_euler(prob::SemilinearODEProblem; dt::Float64) = etd_euler(prob.A, prob.f, prob.u0, prob.tspan, prob.p; dt = dt)
etd_rk2(prob::SemilinearODEProblem; dt::Float64) = etd_rk2(prob.A, prob.f, prob.u0, prob.tspan, prob.p; dt = dt)
etd_rk3(prob::SemilinearODEProblem; dt::Float64) = etd_rk3(prob.A, prob.f, prob.u0, prob.tspan, prob.p; dt = dt)
etd_rk4(prob::SemilinearODEProblem; dt::Float64) = etd_rk4(prob.A, prob.f, prob.u0, prob.tspan, prob.p; dt = dt)

etd_rk4 (generic function with 1 method)

In [4]:
function compare_solutions(prob::SemilinearODEProblem, reference_alg::OrdinaryDiffEqAlgorithm, custom_alg; dt::Real)
    """    
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg    :: one of the following: etd_euler, etd_rk2, etd_rk3, etd_rk4
    """
    #Valid options for `custom_alg`: 

    custom_sol = custom_alg(prob; dt = dt)
    ref_sol = solve(SplitODEProblem(prob), reference_alg; adaptive = false, dt = dt, saveat = custom_sol.t)

    length(custom_sol.t) == length(ref_sol.t) || throw(ArgumentError("custom_sol.t and ref_sol.t have different lengths"))
    
    #STEP 3: Compute difference between reference solution and custom solution 
    u_custom = custom_sol.u
    u_ref = ref_sol.u
    max_l2_error = maximum(norm.(u_ref .- u_custom))

    return (reference_sol = ref_sol, custom_sol = custom_sol, max_l2_error = max_l2_error)     
end 

compare_solutions (generic function with 1 method)

In [21]:
results = run_tests(; title = "ETD tests") do suite

    # reference_alg = NorsettEuler()
    # custom_alg = etd_euler

    # reference_alg = ETDRK2()
    # custom_alg = etd_rk2

    # reference_alg = ETDRK3()
    # custom_alg = etd_rk3

    reference_alg = ETDRK4()
    custom_alg = etd_rk4
    
    tol = 1e-8
    dt = 0.023

    test!(suite, "Riccati") do
        @test compare_solutions(riccati, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Rotation") do
        @test compare_solutions(rotation, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Van der Pol") do
        @test compare_solutions(vanderpol, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end
end;

#Add a couple more tests...

ETD tests
Test          Result      Time (s)
Riccati       PASS        1.66e+00
Rotation      PASS        2.90e+00
Van der Pol   PASS        2.19e+00
──────────────────────────────────
Total                     6.75e+00

Assertions/results: 3 passed · 0 failed · 0 errors
All tests passed.
